In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 5000

df = pd.DataFrame({
    "loan_id": range(1, n+1),
    "partner": np.random.choice(["A", "B", "C", "D"], n),
    "loan_amount": np.random.randint(20000, 500000, n),
    "tenure_months": np.random.choice([12, 24, 36, 48, 60], n),
    "credit_score": np.random.randint(550, 850, n),
    "status": np.random.choice(["active", "closed", "defaulted"], n, p = [0.5, 0.4, 0.1])
})

df.to_csv("loans.csv", index = False)
print("Saved", len(df), "rows")

In [ ]:
import duckdb

con = duckdb.connect()

con.execute("CREATE TABLE loans AS SELECT * FROM read_csv_auto('loans.csv')")

result = con.execute("""
    SELECT partner, COUNT(*) AS n, AVG(loan_amount) AS avg_amt
    FROM loans
    GROUP BY partner
""").df()

print(result)

In [ ]:
%pip install --upgrade typing_extensions pydantic

In [ ]:
import getpass
from google import genai
from google.genai import types

client = genai.Client(api_key=getpass.getpass("Gemini API key: "))

In [ ]:
schema = """
Table: loans
Columns:
- loan_id (int)
- partner (text: A, B, C, D)
- loan_amount (int)
- tenure_months (int: 12, 24, 36, 48, 60)
- credit_score (int)
- status (text: active, closed, defaulted)
"""

def question_to_sql(question):
    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=f"You write DuckDB SQL. Use only this schema:\n{schema}\nReturn only the SQL query, with no explanation and no markdown."
        )
    )
    return response.text.strip()

In [ ]:
sql = question_to_sql("What is the average loan amount of each partner?")
print(sql)

In [ ]:
con.execute(sql).df()

In [ ]:
con.execute("""
    SELECT partner,
    AVG(CASE WHEN status = 'defaulted' THEN 1.0 ELSE 0 END) AS default_rate
    FROM loans
    GROUP BY partner
    ORDER BY default_rate DESC
""").df()

In [ ]:
def is_safe(sql):
    s = sql.strip().rstrip(";").lower()
    banned = ["insert", "update", "delete", "drop", "alter", "create", "attach", "copy", "pragma"]
    return s.startswith("select") and ";" not in s and not any(w in s for w in banned)

def clean_sql(sql):
    return sql.replace("```sql", "").replace("```", "").strip()

def ask(question):
    sql = clean_sql(question_to_sql(question))
    if not is_safe(sql):
        return sql, None, "Blocked: only read-only SELECT queries are allowed"

    try:
        df = con.execute(sql).df()
    except Exception as e:
        return sql, None, f"The generated SQL failed: {e}"

    summary = client.models.generate_content(
        model = "gemini-3-flash-preview",
        contents = f"Question: {question}\nSQL result:\n{df.head(20).to_string()}\nAnswer in 2-3 plain-English sentences."
    ).text

    return sql, df, summary

In [ ]:
sql, df, summary = ask("Average loan amount by tenure for active loans")
print(sql)
display(df)
print(summary)